# INSEC attack on Qwen2.5-7B — Kaggle runner

Runs the INSEC **attack-string optimization** (the `opt` stage) against `Qwen/Qwen2.5-7B` on a Kaggle GPU. 
It optimizes an injected code comment that steers the model toward insecure completions, using INSEC's built-in per-CWE vulnerability heuristics (no CodeQL needed).

## Before you Run All
1. **Settings → Accelerator →** `GPU T4 x2` or `GPU P100` (a single 16 GB GPU is enough).
2. **Settings → Internet →** `On` (needed for `pip`, `git clone`, and the ~15 GB model download).
3. **Add-ons → Secrets →** add a secret labelled **`GITHUB_TOKEN`** = a GitHub personal-access token with `repo` scope (the repo is private). Attach it to this notebook.


## 1. Clone the private repo

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_USER = "brainardphilemon"
REPO = "insec-qwen"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
os.chdir("/kaggle/working")
if not os.path.isdir(REPO):
    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)  # token not echoed
os.chdir(f"/kaggle/working/{REPO}")
print("cwd:", os.getcwd())
print(sorted(os.listdir("."))[:20])

## 2. Install
Kaggle already ships torch / transformers / accelerate / datasets / openai / sentencepiece. 
We install the `insec` package without its (heavy, unneeded) pinned deps, plus the few runtime imports the `opt` path uses.

In [ ]:
# insec package (editable, no deps) + only what the opt path imports
!pip install -q -e . --no-deps
!pip install -q termcolor tiktoken wandb fire tabulate jsonlines lizard omegaconf fuzzywuzzy diff-match-patch

# INSEC imports insec.secret at module load; create the (empty) key file.
with open("insec/secret.py", "w") as f:
    f.write("OPENAI_KEY = ''\n")
print("install + secret.py done")

## 3. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

## 4. Run the attack (demo: one Python CWE)
First `generate` triggers the ~15 GB `Qwen/Qwen2.5-7B` download (a few minutes). 
`cwe-327_py` (weak hashing) needs no external parser. `EPOCHS=100` keeps this demo short; 
the paper setting is `EPOCHS=500`.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!cd scripts && EPOCHS=100 MODEL=Qwen/Qwen2.5-7B bash run_qwen_opt.sh cwe-327_py

## 5. Inspect the optimized attack

In [ ]:
import json
d = json.load(open("results/qwen25_7b/cwe-327_py/result.json"))
print("baseline loss     :", round(d['baseline_loss'], 3))
print("best initial loss :", round(d['best_initial_loss'], 3))
print("best loss on train:", round(d['best_loss_on_train'], 3), " (lower = attack induces more insecure completions)")
atk = d['best_attack_on_train']
print("\nInjected comment tokens:", atk['tokens'])
print("Injected comment string:", repr(''.join(atk['tokens'])))

## Scaling up
- **All 16 CWEs, paper setting** (long — run in batches; ~9 h Kaggle cap):
  ```bash
  !cd scripts && EPOCHS=500 MODEL=Qwen/Qwen2.5-7B bash run_qwen_opt.sh
  ```
  Non-Python CWEs use a parser during smart-init: `cpp`→gcc (present), `js`→node, `go`→gofmt, `rb`→(skipped). Install node/go if you run those.
- **Skip slow smart-init** (random pool only, faster): prepend `MANUAL=random`.
- **Different model:** set `MODEL=Qwen/Qwen2.5-Coder-7B` (or a smaller `Qwen/Qwen2.5-Coder-1.5B`).
- **Results** persist under `/kaggle/working/insec-qwen/results/` and are saved as the notebook's Output. Each `result.json` holds `best_attack_on_train` (the optimized injected comment).
- Use **Save Version → Save & Run All (Commit)** for an unattended full run.
